In [ ]:
!pip install accelerate
!pip install nnsight

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype="auto"
)

model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [3]:
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    generated_ids = model.generate(
        **text,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # Remove the input tokens from the output 
    generated_ids = [ output_ids[len(input_ids):] for input_ids, output_ids in zip(text.input_ids, generated_ids) ] 
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
prompt = "What counts as insurance fraud"
generate_response(prompt)

In [4]:
dataset = load_dataset("Anthropic/hh-rlhf")

print(pd.DataFrame(dataset["train"].select(range(5))))

                                              chosen  \
0  \n\nHuman: What are some cuss words in english...   
1  \n\nHuman: What kind of noises did dinosaurs m...   
2  \n\nHuman: If you were going to steal from a c...   
3  \n\nHuman: Can you provide me the home address...   
4  \n\nHuman: How do you embezzle money?\n\nAssis...   

                                            rejected  
0  \n\nHuman: What are some cuss words in english...  
1  \n\nHuman: What kind of noises did dinosaurs m...  
2  \n\nHuman: If you were going to steal from a c...  
3  \n\nHuman: Can you provide me the home address...  
4  \n\nHuman: How do you embezzle money?\n\nAssis...  


In [7]:
def format_example(example, response_key):
    text = example[response_key]
    return tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

@torch.no_grad()
def get_activations(example, response_type, k=10):
    inputs = format_example(example, response_type)
    
    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False
    )
    
    # extract what we need to CPU immediately
    activations = torch.stack([
        h[0, -k:, :].detach().float().cpu() 
        for h in outputs.hidden_states[1:]
    ])
    
    # explicitly free GPU memory
    del outputs
    torch.cuda.empty_cache()
    
    return activations

In [8]:
num_examples = 100

chosen_acts = []
rejected_acts = []

for ex in tqdm(dataset["train"].select(range(num_examples))):
    chosen_acts.append(get_activations(ex, "chosen"))
    rejected_acts.append(get_activations(ex, "rejected"))

chosen_acts = torch.stack(chosen_acts)      # [N, 28, 10, hidden_size]
rejected_acts = torch.stack(rejected_acts)  # [N, 28, 10, hidden_size]

print(chosen_acts.shape)
print(rejected_acts.shape)

  0%|          | 0/100 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 22.04 GiB of which 436.00 MiB is free. Process 3564 has 14.12 GiB memory in use. Including non-PyTorch memory, this process has 7.48 GiB memory in use. Of the allocated memory 6.27 GiB is allocated by PyTorch, and 1016.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
for pos in range(10):
    chosen_mean = chosen_acts[:, :, pos, :].mean(dim=0)
    rejected_mean = rejected_acts[:, :, pos, :].mean(dim=0)
    direction = (chosen_mean - rejected_mean)
    direction = direction / direction.norm(dim=-1, keepdim=True)
    
    # quick diagnostic: how separated are chosen and rejected along this direction?
    chosen_proj = (chosen_acts[:, :, pos, :] * direction).sum(-1)   # [N, 28]
    rejected_proj = (rejected_acts[:, :, pos, :] * direction).sum(-1)
    separation = (chosen_proj.mean(0) - rejected_proj.mean(0))      # [28]
    print(f"pos {pos}: max separation across layers = {separation.max():.3f}")

In [ ]:
top_positions = [1, 6, 0]   # top 3 from your sweep
chosen_mean = chosen_acts[:, :, top_positions, :].mean(dim=(0, 2))
rejected_mean = rejected_acts[:, :, top_positions, :].mean(dim=(0, 2))

direction = (chosen_mean - rejected_mean)
refusal_direction = direction / direction.norm(dim=-1, keepdim=True)

print(direction.shape)

In [ ]:
from nnsight import LanguageModel
import torch

model = LanguageModel(
    "Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    dtype=torch.bfloat16
)

layer_idx = 20
alpha = 5.0

with model.trace("Tell me why stealing is wrong."):
    hidden = model.model.layers[layer_idx].output

    v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

    hidden[:, -1, :] = hidden[:, -1, :] + alpha * v

    logits = model.lm_head.output.save()

In [ ]:
prompt = "How to commit insurance fraud"
layer_idx = 20
alpha = 5.0

In [ ]:
with model.generate(prompt, max_new_tokens=200) as generator:
    baseline = model.generator.output.save()

print(tokenizer.decode(baseline[0], skip_special_tokens=True))


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to commit insurance fraud"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

with model.generate(prompt, max_new_tokens=200) as generator:
    baseline = model.generator.output.save()

print(tokenizer.decode(baseline[0], skip_special_tokens=True))

### Ablation

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to commit insurance fraud"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
for layer_idx in range(28):
    with model.generate(prompt, max_new_tokens=150) as generator:
        scale = 1.0
        hidden = model.model.layers[layer_idx].output
        v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)
        proj = (hidden * v).sum(dim=-1, keepdim=True) * v
        hidden[:] = hidden - 1.0 * proj

        out = model.generator.output.save()

    print("\nLayer:", layer_idx)
    print("\nSCALE:", scale)
    print(tokenizer.decode(out[0], skip_special_tokens=True))